# Project 8: Plant Growth Parameter Estimation

In this notebook, you will implement the plant-growth model dynamics and objective function, then run GA training and evaluate performance on a held-out season.

## Part 1: Generate a dataset

1. Complete the FILL IN THE BLANK sections in `plantga/simulation.py` for:
   - `compute_pairwise_distances`
   - `interaction_kernel`
   - model dynamics update in `simulate_heights`
2. The input functions (irrigation, fertilizer, sunlight) and GA are provided in full.
3. Run the cells to inspect generated trajectories and include your plot in your submission.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from write_parameters import default_parameters
from plantga.data_generation import generate_dataset
from plantga.simulation import simulate_heights

params = default_parameters()
dataset = generate_dataset(true_params=params['true_params'], **params['dataset'])

print(f"Plants: {dataset['positions'].shape[0]}")
print(f"Time steps per season: {len(dataset['time'])}")
print(f"Training season observations: {dataset['h_obs'].shape}")
print(f"Held-out test season observations: {dataset['h_obs_test'].shape}")

In [ ]:
time = dataset['time']

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)

train_mean_clean = dataset['h_clean'].mean(axis=1)
train_mean_obs = dataset['h_obs'].mean(axis=1)

axes[0].plot(time, train_mean_clean, color='tab:blue', linewidth=2, label='Clean mean')
axes[0].plot(time, train_mean_obs, color='tab:blue', linestyle='--', linewidth=2, label='Observed mean')
axes[0].set_title('Generated Trajectory: Train Season')
axes[0].set_xlabel('Day')
axes[0].set_ylabel('Height')
axes[0].grid(alpha=0.25)

test_mean_clean = dataset['h_clean_test'].mean(axis=1)
test_mean_obs = dataset['h_obs_test'].mean(axis=1)

axes[1].plot(time, test_mean_clean, color='tab:orange', linewidth=2, label='Clean mean')
axes[1].plot(time, test_mean_obs, color='tab:orange', linestyle='--', linewidth=2, label='Observed mean')
axes[1].set_title('Generated Trajectory: Held-out Test Season')
axes[1].set_xlabel('Day')
axes[1].grid(alpha=0.25)

axes[0].legend(fontsize=8)
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

## Part 2: Visualize seasonal forcing

In [ ]:
time = dataset['time']

fig, ax = plt.subplots(3, 2, figsize=(12, 7), sharex=True)

ax[0, 0].plot(time, dataset['W'], color='tab:blue')
ax[0, 0].set_title('Season 1 Forcing (Train)')
ax[0, 0].set_ylabel('W(t)')

ax[0, 1].plot(time, dataset['W_test'], color='tab:blue')
ax[0, 1].set_title('Season 2 Forcing (Held-out Test)')

ax[1, 0].stem(time, dataset['F'], linefmt='tab:green', markerfmt='go', basefmt='k-')
ax[1, 0].set_ylabel('F(t)')

ax[1, 1].stem(time, dataset['F_test'], linefmt='tab:green', markerfmt='go', basefmt='k-')

ax[2, 0].plot(time, dataset['S'], color='tab:orange')
ax[2, 0].set_ylabel('S(t)')
ax[2, 0].set_xlabel('Day')

ax[2, 1].plot(time, dataset['S_test'], color='tab:orange')
ax[2, 1].set_xlabel('Day')

for col in range(2):
    for row in range(3):
        ax[row, col].grid(alpha=0.2)

plt.tight_layout()
plt.show()

## Part 3: Run the genetic algorithm
Complete the FILL IN THE BLANK section in `plantga/genetic_algorithm.py` for the objective function (`normalized_mse`).

If this cell fails, revisit the FILL IN THE BLANK sections in `plantga/simulation.py` and `plantga/genetic_algorithm.py`.

In [ ]:
from plantga.genetic_algorithm import GeneticAlgorithmEstimator, normalized_mse

ga_cfg = dict(params['ga'])

estimator = GeneticAlgorithmEstimator(bounds=params['bounds'], **ga_cfg)
result = estimator.run(dataset)

print(f"Best train cost: {result['best_train_cost']:.4f}")

## Part 4: Analyze parameter recovery

In [ ]:
print('Parameter recovery summary')
print('-' * 64)
print(f"{'param':<12}{'true':>12}{'estimated':>14}{'abs error':>14}")
print('-' * 64)
for name in result['param_names']:
    true_v = params['true_params'][name]
    est_v = result['best_params'][name]
    err = abs(est_v - true_v)
    print(f"{name:<12}{true_v:>12.4f}{est_v:>14.4f}{err:>14.4f}")

h_pred_train = simulate_heights(
    params=result['best_params'],
    initial_heights=dataset['initial_heights'],
    time=dataset['time'],
    W=dataset['W'],
    F=dataset['F'],
    S=dataset['S'],
    positions=dataset['positions'],
)

h_pred_test = simulate_heights(
    params=result['best_params'],
    initial_heights=dataset['initial_heights_test'],
    time=dataset['time'],
    W=dataset['W_test'],
    F=dataset['F_test'],
    S=dataset['S_test'],
    positions=dataset['positions'],
)

train_mse = normalized_mse(dataset['h_obs'][dataset['train_idx']], h_pred_train[dataset['train_idx']])
test_mse = normalized_mse(dataset['h_obs_test'], h_pred_test)
print(f"Training objective (normalized MSE): {train_mse:.4f}")
print(f"Held-out test objective (normalized MSE): {test_mse:.4f}")

train_mean_actual = dataset['h_obs'].mean(axis=1)
train_mean_pred = h_pred_train.mean(axis=1)
test_mean_actual = dataset['h_obs_test'].mean(axis=1)
test_mean_pred = h_pred_test.mean(axis=1)

train_mean_mse = float(np.mean((train_mean_actual - train_mean_pred) ** 2))
test_mean_mse = float(np.mean((test_mean_actual - test_mean_pred) ** 2))
print(f"Train mean-trajectory MSE: {train_mean_mse:.6f}")
print(f"Held-out test mean-trajectory MSE: {test_mean_mse:.6f}")

candidate_indices = [0, dataset['positions'].shape[0] - 1]
plant_indices = sorted(set(candidate_indices))
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)

for idx in plant_indices:
    axes[0].plot(dataset['time'], dataset['h_obs'][:, idx], linestyle=':', alpha=0.55, label=f'Observed plant {idx}')
    axes[0].plot(dataset['time'], h_pred_train[:, idx], linewidth=1.8, alpha=0.9, label=f'Predicted plant {idx}')

    axes[1].plot(dataset['time'], dataset['h_obs_test'][:, idx], linestyle=':', alpha=0.55, label=f'Observed plant {idx}')
    axes[1].plot(dataset['time'], h_pred_test[:, idx], linewidth=1.8, alpha=0.9, label=f'Predicted plant {idx}')

axes[0].plot(dataset['time'], train_mean_actual, color='black', linestyle=':', linewidth=2.3, label='Actual mean')
axes[0].plot(dataset['time'], train_mean_pred, color='black', linewidth=2.3, label='Predicted mean')
axes[1].plot(dataset['time'], test_mean_actual, color='black', linestyle=':', linewidth=2.3, label='Actual mean')
axes[1].plot(dataset['time'], test_mean_pred, color='black', linewidth=2.3, label='Predicted mean')

axes[0].set_title('Season 1 (Train): Plants + Mean')
axes[1].set_title('Season 2 (Held-out Test): Plants + Mean')
axes[0].set_xlabel('Day')
axes[1].set_xlabel('Day')
axes[0].set_ylabel('Height')
axes[0].grid(alpha=0.2)
axes[1].grid(alpha=0.2)
axes[0].legend(ncol=1, fontsize=8, loc='upper right')
axes[1].legend(ncol=1, fontsize=8, loc='upper right')

plt.tight_layout()
plt.show()